# 15 · Ek CPU gecikme ölçümü ve teslim paketi

**Çalışma zamanı: CPU** (GPU gerekmez; Çalışma zamanı → Çalışma zamanı türünü değiştir → CPU). Süre yaklaşık 10-15 dk. Notebook 11-12 çalışmış olmalı (model ağırlıkları `outputs/runs/` içinde).

1. Ablasyon varyantları ve kontrol modeli için CPU gecikmesi (batch 1). Karşılaştırılabilirlik için tam model, ConvNeXt-Tiny ve DeiT-Small aynı oturumda yeniden ölçülür.
2. Dağıtım paketi: `deploy/` içinde float16 ağırlık kopyası (GitHub 100 MB sınırı için) ve float32 ile doğruluk kontrolü
3. `predict.py` denemesi (CPU)
4. Ortam kaydı (`pip freeze`, sürümler) ve çıktıların zip'lenmesi

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
# Kod ve çıktılar Drive'da tek klasörde durur; ham veri oturum diskinde (/content/data) tutulur.
PROJECT = next((p for p in ('/content/drive/MyDrive/YZProje', '/content/drive/MyDrive/YZProje/YZProje')
                if os.path.isfile(p + '/src/config.py')), None)
assert PROJECT, "Drive'da YZProje/src/config.py bulunamadı: proje klasörünü MyDrive/YZProje olarak yerleştirin."
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = PROJECT + '/outputs'
os.environ['KNEE_DATA_ROOT'] = '/content/data'
print('Proje klasörü:', PROJECT)

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0 timm==1.0.30 scikit-image
import torch, platform
print(torch.__version__, platform.python_version(), 'GPU' if torch.cuda.is_available() else 'CPU')

In [ ]:
# Görüntüler oturum diskinde yoksa: önce Drive'daki arşiv kopyası, o da yoksa bağlantıdan indirip arşivi Drive'a kaydeder.
from src import config as C, data_prep
print('Veri hazır:', data_prep.ensure_data())

## 1 · CPU gecikme (ablasyon + kontrol)

In [ ]:
import os, platform, subprocess
print(platform.processor() or platform.machine(), '| CPU sayısı:', os.cpu_count())
print(subprocess.run(['bash','-c','lscpu | grep -E "Model name|Thread|Core"'], capture_output=True, text=True).stdout)
from src import final_eval as F
df = F.cpu_latency_extra()
df[['model','params','latency_ms_median','latency_ms_p95','round_medians']].round(1)

## 2 · Dağıtım paketi (float16 kopya + doğruluk kontrolü)

In [ ]:
from src import export
print(export.export())          # model.pt + meta.json (float32)
print(export.export_fp16())      # model_fp16.pt ve val üzerinde karşılaştırma

## 3 · predict.py denemesi

In [ ]:
from src.data import read_split
from src.data_prep import get_dataset_root
r = read_split('val').iloc[3]
!python predict.py "{get_dataset_root() / r.relpath}"
print('gerçek derece:', r.label)

## 4 · Ortam kaydı ve çıktıları kaydet

In [ ]:
import shutil, json, subprocess
from src import config as C
(C.OUTPUT_DIR / 'env').mkdir(parents=True, exist_ok=True)
(C.OUTPUT_DIR / 'env' / 'pip_freeze_final.txt').write_text(subprocess.run(['pip','freeze'], capture_output=True, text=True).stdout)
shutil.copytree(C.PROJECT_ROOT / 'deploy', '/content/deploy_out', dirs_exist_ok=True)
shutil.make_archive('/content/gecikme_ve_deploy', 'zip', '/content/deploy_out')
import pandas as pd
z = C.OUTPUT_DIR / 'final'
shutil.copy(z / 'cpu_latency_extra.csv', '/content/cpu_latency_extra.csv')
print('hazır: /content/gecikme_ve_deploy.zip ve /content/cpu_latency_extra.csv -> ikisini de indirip bana gönderin')
from google.colab import files
files.download('/content/cpu_latency_extra.csv'); files.download('/content/gecikme_ve_deploy.zip')

## 5 · Teslim klasörü (en son, rapor son hâlini aldıktan sonra)

Rapor dosyalarını (`YZProje_Rapor_*.docx/.pdf`) Drive'da `YZProje/report/` klasörüne koyun; sonra bu hücreyi çalıştırın. Çıktı: `220709071_BahaTurk_YZProje.zip` (README'deki SHA256 otomatik doldurulur).

In [ ]:
!python tools/make_submission.py --project {C.PROJECT_ROOT} --out /content
from google.colab import files
files.download('/content/220709071_BahaTurk_YZProje.zip')